# 蛋白–生物素化多肽 MD 指标分析（单变体）

输入：`out_biop_<VARIANT>/`（md.tpr + md_*.xtc + md_*.edr）
输出：`ANALYSIS/<VARIANT>/` 下的 xvg、csv、figs

指标：
1. 结合保持率（肽位移 + 接触占时比）
2. 279 位侧链 ⇄ PEB / BTN 环 的距离分布
3. 界面埋藏面积（buried SASA）
4. 分子间氢键（占时率）
5. RMSD / Rg / RMSF（结构稳定性）

第 6 节 MM-GBSA 需要 gmx_MMPBSA 环境，默认 `SKIP`。

In [ ]:
import os, glob, subprocess, shutil, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ========================= CONFIG =========================
VARIANT = "Y262A"                                  # 改这里：WT / Y262A / W282A
BASE    = "/root/autodl-tmp/GROMACS"
GMX     = "/root/gromacs_cuda/bin/gmx"
if not os.path.exists(GMX):                        # 412 上 gmx 曾被移到 dgl-master 下
    for _c in (glob.glob("/root/*/gromacs_cuda/bin/gmx") +
               glob.glob("/root/autodl-tmp/*/bin/gmx")):
        GMX = _c
        break
RUN_MMPBSA = False                                 # 有 gmx_MMPBSA 环境后再设 True
# ==========================================================

D   = f"{BASE}/{VARIANT}/out_biop_{VARIANT}"
OUT = f"{BASE}/ANALYSIS/{VARIANT}"
os.makedirs(f"{OUT}/figs", exist_ok=True)
os.makedirs(f"{OUT}/logs", exist_ok=True)
os.makedirs(f"{OUT}/xvg",  exist_ok=True)

TPR = f"{D}/md.tpr"
XTC = sorted(glob.glob(f"{D}/md_*.part0*.xtc") or glob.glob(f"{D}/md_*.xtc"))[0]
EDR = sorted(glob.glob(f"{D}/md_*.edr"))[0]
IDX = f"{OUT}/index.ndx"
for f in (TPR, XTC, EDR):
    assert os.path.exists(f), f"missing {f}"
print("variant :", VARIANT)
print("traj    :", XTC, f"{os.path.getsize(XTC)/1e9:.2f} GB")
print("out     :", OUT)

In [ ]:
def run(args, stdin_text=None, tag="run"):
    r = subprocess.run([GMX] + args, input=stdin_text, text=True, capture_output=True)
    with open(f"{OUT}/logs/{tag}.log", "w") as f:
        f.write(r.stdout + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{tag} failed:\n" + (r.stdout + r.stderr)[-1500:])
    return r

def read_xvg(path):
    rows = []
    for ln in open(path):
        s = ln.strip()
        if not s or s[0] in "#@":
            continue
        rows.append([float(x) for x in s.split()])
    return np.array(rows)

# --- default index groups (Protein / Backbone / PEB ...) ---
run(["make_ndx", "-f", TPR, "-o", IDX], stdin_text="q\n", tag="makenndx")
GROUPS = {}
for ln in open(f"{OUT}/logs/makenndx.log"):
    p = ln.split()
    if len(p) >= 4 and p[0].isdigit() and p[1] != ":":
        name = ln.split(":")[0].split(None, 1)[1].strip()
        GROUPS[name] = int(p[0])
print(GROUPS)

## 1. 结构稳定性：RMSD / Rg / RMSF

- **RMSD** 证明突变没把结构弄坏
- **Rg** 排除“解聚/解折叠”这种伪阳性
- **RMSF** 看 262/279/282 附近柔性是否变化

In [ ]:
sel = f"{GROUPS['Backbone']}\n{GROUPS['Backbone']}\n"
run(["-quiet","rms","-s",TPR,"-f",XTC,"-n",IDX,"-o",f"{OUT}/xvg/rmsd_bb.xvg","-tu","ns"], stdin_text=sel, tag="rmsd")
d = read_xvg(f"{OUT}/xvg/rmsd_bb.xvg")
rmsd = pd.DataFrame({"t_ns": d[:,0], "rmsd_nm": d[:,1]})

run(["-quiet","gyrate","-s",TPR,"-f",XTC,"-n",IDX,"-o",f"{OUT}/xvg/gyrate.xvg"], stdin_text=f"{GROUPS['Protein']}\n", tag="gyrate")
g = read_xvg(f"{OUT}/xvg/gyrate.xvg")
rg = pd.DataFrame({"t_ns": g[:,0], "Rg_nm": g[:,1]})

run(["-quiet","rmsf","-s",TPR,"-f",XTC,"-n",IDX,"-o",f"{OUT}/xvg/rmsf.xvg","-res"], stdin_text=f"{GROUPS['C-alpha']}\n", tag="rmsf")
fu = read_xvg(f"{OUT}/xvg/rmsf.xvg")
rmsf = pd.DataFrame({"res": fu[:,0], "rmsf_nm": fu[:,1]})

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].plot(rmsd.t_ns, rmsd.rmsd_nm, lw=0.8); ax[0].set(xlabel="time (ns)", ylabel="backbone RMSD (nm)", title=f"{VARIANT}: RMSD")
ax[1].plot(rg.t_ns, rg.Rg_nm, lw=0.8, color="C1"); ax[1].set(xlabel="time (ns)", ylabel="Rg (nm)", title="Rg")
ax[2].plot(rmsf.res, rmsf.rmsf_nm, lw=0.8, color="C2"); ax[2].set(xlabel="residue", ylabel="RMSF (nm)", title="C-alpha RMSF")
for r in (262, 279, 282):
    ax[2].axvline(r, color="0.6", ls=":", lw=0.8)
plt.tight_layout(); plt.savefig(f"{OUT}/figs/01_stability.png", dpi=150)
print("rmsd end %.3f nm | Rg mean %.3f nm" % (rmsd.rmsd_nm.iloc[-1], rg.Rg_nm.mean()))

## 2. 结合保持率（闸门）

- **接触占时比**：Protein 与 PEB 有原子对 < 4 Å 的帧占比
- **肽位移**：拟合蛋白后，肽质心的位移

占时比 > 80% → 结合保持；< 20% 且肽漂远 → 解离，后面的 ΔG 不必算。

In [ ]:
run(["-quiet","mindist","-s",TPR,"-f",XTC,"-n",IDX,"-od",f"{OUT}/xvg/mindist_peb.xvg",
     "-on",f"{OUT}/xvg/ncontacts.xvg","-d","0.4"],
    stdin_text=f"{GROUPS['Protein']}\n{GROUPS['PEB']}\n", tag="mindist")
m  = read_xvg(f"{OUT}/xvg/mindist_peb.xvg")
nc = read_xvg(f"{OUT}/xvg/ncontacts.xvg")
contact = pd.DataFrame({"t_ns": nc[:,0], "n_contacts_4A": nc[:,1]})
occ = float((contact.n_contacts_4A > 0).mean())
print(f"contact occupancy (>=1 pair <4A): {occ*100:.1f}%  | min-dist mean {m[:,1].mean():.3f} nm")

# peptide RMSD after fitting protein
run(["-quiet","rms","-s",TPR,"-f",XTC,"-n",IDX,"-o",f"{OUT}/xvg/rmsd_peb.xvg","-tu","ns"],
    stdin_text=f"{GROUPS['Protein']}\n{GROUPS['PEB']}\n", tag="rmsd_pep")
p = read_xvg(f"{OUT}/xvg/rmsd_peb.xvg")
pep = pd.DataFrame({"t_ns": p[:,0], "rmsd_nm": p[:,1]})
# 周期性边界跳变会给出 8-12 nm 的孤立尖峰，剔除以避免污染均值
n_out = int((pep.rmsd_nm > 2.0).sum())
pep_clean = pep[pep.rmsd_nm <= 2.0]
print(f"peptide displacement: median {pep_clean.rmsd_nm.median():.3f} nm | "
      f"p95 {pep_clean.rmsd_nm.quantile(.95):.3f} nm | PBC outliers dropped: {n_out}")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].plot(contact.t_ns, contact.n_contacts_4A, lw=0.6, color="C3"); ax[0].set(xlabel="time (ns)", ylabel="# atom pairs < 4 A", title=f"interface contacts (occ {occ*100:.0f}%)")
ax[1].plot(m[:,0], m[:,1], lw=0.6); ax[1].set(xlabel="time (ns)", ylabel="min distance (nm)", title="Protein-PEB min distance")
ax[2].plot(pep_clean.t_ns, pep_clean.rmsd_nm, lw=0.6, color="C4"); ax[2].set(xlabel="time (ns)", ylabel="peptide RMSD (nm)", title=f"peptide displacement (median {pep_clean.rmsd_nm.median():.2f} nm)")
plt.tight_layout(); plt.savefig(f"{OUT}/figs/02_binding_retention.png", dpi=150)

## 3. 279 位侧链 ⇄ PEB / BTN 环 的距离分布

- **统一量**（WT/R279E 可比）：`resnr 279` 与 `resname PEB` 的最小距离
- **相对位置**：279 位侧链到 PEB 的最近距离分布 + 直方图

注意：BTN 在 PEB 里的原子编号依赖 tpr 的原子顺序，本 notebook 用 “PEB 分子起点 + 29” 定位 BTN 重原子（PEB 前 29 个重原子是肽）。

In [ ]:
run(["-quiet","pairdist","-s",TPR,"-f",XTC,"-ref","resnr 279","-sel","resname PEB",
     "-o",f"{OUT}/xvg/dist_279_peb.xvg"], tag="pairdist279")
dd = read_xvg(f"{OUT}/xvg/dist_279_peb.xvg")
dist279 = pd.DataFrame({"t_ns": dd[:,0], "d_nm": dd[:,1]})

# BTN (biotin) atom range inside PEB: peptide has 29 heavy atoms -> BTN = next 15
n_prot = m.shape[0] and None
prot_atoms = int([l for l in open(f"{OUT}/logs/makenndx.log") if "Protein " in l][0].split(":")[1].split()[0])
btn_lo, btn_hi = prot_atoms + 30, prot_atoms + 44     # 1-based inclusive
run(["-quiet","pairdist","-s",TPR,"-f",XTC,"-ref","resnr 279","-sel",f"atomnr {btn_lo} to {btn_hi}",
     "-o",f"{OUT}/xvg/dist_279_btn.xvg"], tag="pairdist_btn")
bb = read_xvg(f"{OUT}/xvg/dist_279_btn.xvg")
distbtn = pd.DataFrame({"t_ns": bb[:,0], "d_nm": bb[:,1]})

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].plot(dist279.t_ns, dist279.d_nm, lw=0.6); ax[0].set(xlabel="time (ns)", ylabel="min dist (nm)", title="res 279 <-> PEB")
ax[1].plot(distbtn.t_ns, distbtn.d_nm, lw=0.6, color="C5"); ax[1].set(xlabel="time (ns)", ylabel="min dist (nm)", title="res 279 <-> BTN")
ax[2].hist(dist279.d_nm, bins=40, alpha=.6, density=True, label="PEB")
ax[2].hist(distbtn.d_nm, bins=40, alpha=.6, density=True, label="BTN")
ax[2].set(xlabel="min distance (nm)", ylabel="density", title="distance distribution"); ax[2].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/figs/03_distance_279.png", dpi=150)

stat = pd.DataFrame({
    "quantity": ["d279_PEB_mean", "d279_PEB_p05", "d279_PEB_p95", "d279_BTN_mean"],
    "value_nm": [dist279.d_nm.mean(), dist279.d_nm.quantile(.05), dist279.d_nm.quantile(.95), distbtn.d_nm.mean()]})
stat.to_csv(f"{OUT}/distance_stats.csv", index=False)
print(stat.to_string(index=False))

## 4. 界面埋藏面积（buried SASA）

复杂物 SASA 减去两个单体 SASA，得到界面埋藏面积。突变若削弱结合，这个值会下降。

In [ ]:
SEL_PEP  = 'group "PEB"'
SEL_PROT = 'group "Protein"'
SEL_COMP = 'group "Protein" or group "PEB"'

def sasa_total(sel, tag):
    run(["sasa","-s",TPR,"-f",XTC,"-n",IDX,"-surface",sel,"-dt","0.5",
         "-o",f"{OUT}/xvg/{tag}.xvg"], tag=tag)
    a = read_xvg(f"{OUT}/xvg/{tag}.xvg")
    return pd.DataFrame({"t_ns": a[:,0], "area_nm2": a[:,1]})

try:
    sc = sasa_total(SEL_COMP, "sasa_complex")
    sp = sasa_total(SEL_PROT, "sasa_protein")
    sl = sasa_total(SEL_PEP,  "sasa_peb")
    n = min(len(sc), len(sp), len(sl))
    buried = pd.DataFrame({"t_ns": sc.t_ns[:n],
                           "buried_nm2": (sp.area_nm2[:n] + sl.area_nm2[:n] - sc.area_nm2[:n])})
    buried.to_csv(f"{OUT}/buried_sasa.csv", index=False)
    print("buried SASA mean %.2f nm2" % buried.buried_nm2.mean())
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.plot(buried.t_ns, buried.buried_nm2, lw=.7, color="C7")
    ax.set(xlabel="time (ns)", ylabel="buried SASA (nm2)", title="interface buried area")
    plt.tight_layout(); plt.savefig(f"{OUT}/figs/04_buried_sasa.png", dpi=150)
except Exception as e:
    print("SASA step failed (non-fatal):", str(e)[-400:])

# 逐残基 SASA（Protein 单独算），用于侧链埋藏比
try:
    run(["sasa","-s",TPR,"-f",XTC,"-n",IDX,"-surface",SEL_PROT,"-dt","1.0",
         "-o",f"{OUT}/xvg/sasa_prot_only.xvg","-or",f"{OUT}/xvg/sasa_res.xvg"], tag="sasa_res")
    print("per-residue SASA written -> xvg/sasa_res.xvg")
except Exception as e:
    print("per-residue SASA failed (non-fatal):", str(e)[-300:])

## 5. 分子间氢键

只统计 Protein ↔ PEB 之间的氢键，输出每帧数量与总占时率。

In [ ]:
# PEB 用 ACPYPE 的 GAFF 原子名（N1/O3/H93），tpr 里没有元素类型，gmx hbond 判定不了供体/受体。
# 改用几何判据：「PEB 的 N/O 与蛋白的 N/O 在 3.5 Å 内的原子对数目」= 候选氢键数。
# 实现放在 scripts/hbond_geometric.py（已验证可跑），这里只调用并读回结果。
HB_SCRIPT = "/root/autodl-tmp/GROMACS/ANALYSIS/scripts/hbond_geometric.py"
try:
    r = subprocess.run([sys.executable, HB_SCRIPT, VARIANT], capture_output=True, text=True)
    print(r.stdout[-500:])
    if r.returncode != 0:
        print(r.stderr[-500:])
    pc = read_xvg(f"{OUT}/xvg/polar_pairs.xvg")
    pol = pd.DataFrame({"t_ns": pc[:,0], "n_polar_pairs": pc[:,1]})
    hb_mean = float(pol.n_polar_pairs.mean())
    print("candidate H-bond pairs (N/O..N/O < 3.5 A): mean %.1f | median %.1f | max %d"
          % (hb_mean, pol.n_polar_pairs.median(), pol.n_polar_pairs.max()))
except Exception as e:
    pol = pd.DataFrame({"n_polar_pairs": [float("nan")]})
    hb_mean = float("nan")
    print("polar-pair step failed (non-fatal):", str(e)[-400:])

## 6. 汇总

输出 `summary.csv`：一行为本变体。后续把各变体的 csv 合并就是总表。

In [ ]:
summary = pd.DataFrame([{
    "variant": VARIANT,
    "atoms": int(open(f"{OUT}/logs/makenndx.log").read().split("System")[1].split(":")[1].split()[0]),
    "ns_done": float(rmsd.t_ns.iloc[-1]),
    "rmsd_end_nm": float(rmsd.rmsd_nm.iloc[-1]),
    "rmsd_max_nm": float(rmsd.rmsd_nm.max()),
    "Rg_mean_nm": float(rg.Rg_nm.mean()),
    "contact_occupancy": occ,
    "min_dist_mean_nm": float(m[:,1].mean()),
    "pep_rmsd_median_nm": float(pep_clean.rmsd_nm.median()),
    "pep_pbc_outliers": n_out,
    "d279_PEB_mean_nm": float(dist279.d_nm.mean()),
    "d279_BTN_mean_nm": float(distbtn.d_nm.mean()),
    "candidate_hbonds_mean": hb_mean,
}])
summary.to_csv(f"{OUT}/summary.csv", index=False)
print(summary.T.to_string())
print("\nfigs:", sorted(os.listdir(f"{OUT}/figs")))

## 7. MM-GBSA（可选）

需要 `gmx_MMPBSA` 环境（含 AmberTools）。把 `RUN_MMPBSA` 改成 `True` 才会执行。

```bash
gmx_MMPBSA -O -i mmpbsa.in -cs md.tpr -ci index.ndx -cg 1 2 -ct md_Y262A.xtc -eo final.dat
```

`mmpbsa.in` 示例（同时算 GB 与 PB）：

```
&general
  sys_name="VAR", startframe=101, endframe=1000, interval=5, verbose=2,
/
&gb
  igb=5, saltcon=0.15,
/
&pb
  istrng=0.15,
/
&decomp
  idecomp=2, dec_verbose=1, print_res="all",
/
```

In [ ]:
if RUN_MMPBSA:
    mmpbsa = os.path.expanduser("~/mmpbsa/bin/gmx_MMPBSA")
    inp = f"{OUT}/mmpbsa.in"
    with open(inp, "w") as f:
        f.write('&general\n  sys_name="%s", startframe=1, endframe=99999, interval=5, verbose=2,\n/\n' % VARIANT)
        f.write('&gb\n  igb=5, saltcon=0.15,\n/\n&pb\n  istrng=0.15,\n/\n')
        f.write('&decomp\n  idecomp=2, dec_verbose=1, print_res="all",\n/\n')
    r = subprocess.run([mmpbsa, "-O", "-i", inp, "-cs", TPR, "-ci", IDX, "-cg",
                        str(GROUPS["Protein"]), str(GROUPS["PEB"]), "-ct", XTC,
                        "-eo", f"{OUT}/mmpbsa_final.dat", "-o", f"{OUT}/mmpbsa.csv"],
                       text=True, capture_output=True)
    print(r.stdout[-2000:])
else:
    print("RUN_MMPBSA = False, skipped")